> 在线讲义：[第 09 章 · KV 怎么变小 ①：头共享与潜变量压缩](https://wikieden.github.io/ai-infra-from-constraints/lessons/09-kv-sharing.html) · 本书《从约束推导 AI Infra》

# 第 9 章 · KV 怎么变小 ①：头共享与潜变量压缩

配套讲义：仓库里 `ai-infra/notes/lessons/09-kv-sharing.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 Qwen3-8B 三种结构 | $2Ln_{\mathrm{KV}}d_hbH$ | `kv_gib(n_kv)` | 4.5、1.125、0.140625 GiB |
| 第二步 注意力运算量 | $4n_Qd_hN_{\mathrm{pair}}$ | `attn_flops_per_pair(n_q=32)` | 16384（与 KV 头数无关） |
| 第三步 小例子 | 两条路径 | `scores_expanded(c, U, q)`、`scores_compact(c, U, q)` | [3, 8] 和 [3, 8] |
| 第三步 Kimi K3 | $24 \times 8192 \times 576 \times 2$ | `mla_compact_mib(8192)`、`mla_expanded_gib(8192)` | 216 MiB、11.25 GiB |

Qwen3-8B（36 层、头宽 128、32 个查询头）与 Kimi K3（24 个 MLA 层、512 + 64 维、展开后 1,474,560 B/token）取自 AII 第 2 章 §2.3.1–§2.3.2；
c、U、q 的小矩阵是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `kv_gib`**：H 个 token 的 KV（GiB）= 2 × L × n_kv × d_h × b × H。

In [ ]:
def kv_gib(n_kv: int, L: int = 36, d_h: int = 128, H: int = 8192, b: int = 2) -> float:
    """H 个 token 的 KV（GiB）= 2 × L × n_kv × d_h × b × H。"""
    return 2 * L * n_kv * d_h * b * H / 2**30

**函数 `attn_flops_per_pair`**：每个查询—键配对的注意力运算：4 × n_q × d_h，与 KV 头数无关。

In [ ]:
def attn_flops_per_pair(n_q: int = 32, d_h: int = 128) -> int:
    """每个查询—键配对的注意力运算：4 × n_q × d_h，与 KV 头数无关。"""
    return 4 * n_q * d_h

**函数 `matmul`**：普通矩阵乘法（列表实现，方便看清每一步）。

In [ ]:
def matmul(a: list[list[float]], b: list[list[float]]) -> list[list[float]]:
    """普通矩阵乘法（列表实现，方便看清每一步）。"""
    return [[sum(a[i][t] * b[t][j] for t in range(len(b))) for j in range(len(b[0]))] for i in range(len(a))]

**函数 `transpose`**：转置。

In [ ]:
def transpose(a: list[list[float]]) -> list[list[float]]:
    """转置。"""
    return [list(r) for r in zip(*a)]

**函数 `scores_expanded`**：展开路径：K = cU，每个上下文 token 都升维，再算 qKᵀ。

In [ ]:
def scores_expanded(c: list[list[float]], U: list[list[float]], q: list[float]) -> list[float]:
    """展开路径：K = cU，每个上下文 token 都升维，再算 qKᵀ。"""
    K = matmul(c, U)
    return matmul([q], transpose(K))[0]

**函数 `scores_compact`**：紧凑路径：先算 qUᵀ（只变换当前查询），再和潜变量 c 点积。

In [ ]:
def scores_compact(c: list[list[float]], U: list[list[float]], q: list[float]) -> list[float]:
    """紧凑路径：先算 qUᵀ（只变换当前查询），再和潜变量 c 点积。"""
    qU = matmul([q], transpose(U))[0]
    return matmul([qU], transpose(c))[0]

**函数 `mla_compact_mib`**：MLA 紧凑缓存（MiB）= 层数 × H × (d_c + d_r) × b。

In [ ]:
def mla_compact_mib(H: int, layers: int = 24, d_c: int = 512, d_r: int = 64, b: int = 2) -> float:
    """MLA 紧凑缓存（MiB）= 层数 × H × (d_c + d_r) × b。"""
    return layers * H * (d_c + d_r) * b / 2**20

**函数 `mla_expanded_gib`**：MLA 展开缓存（GiB）= H × 每 token 展开后的字节数。

In [ ]:
def mla_expanded_gib(H: int, per_token: int = 1_474_560) -> float:
    """MLA 展开缓存（GiB）= H × 每 token 展开后的字节数。"""
    return H * per_token / 2**30

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步：Qwen3-8B 只改 KV 头数，8192 个 token 的 KV")
for name, n_kv in (("MHA", 32), ("GQA", 8), ("MQA", 1)):
    print(f"  {name}：{n_kv} 个 KV 头 → {kv_gib(n_kv):.6g} GiB")
print(f"  注意力每个配对 {attn_flops_per_pair()} FLOPs，三种结构都一样")

In [ ]:
print("\n第三步：结合律小例子（编的）")
c, U, q = [[1, 0], [2, 1]], [[1, 0, 2], [0, 1, 1]], [1, 1, 1]
print(f"  展开 {scores_expanded(c, U, q)}，紧凑 {scores_compact(c, U, q)}")

In [ ]:
print("\n第三步：Kimi K3 的 24 个 MLA 层，8192 个 token，BF16")
print(f"  紧凑 {mla_compact_mib(8192):.0f} MiB，展开 {mla_expanded_gib(8192):.2f} GiB")

## 3. 代码里才看得到的

1. 小例子用整数，两条路径结果完全相等；真实浮点计算顺序不同会有舍入误差，场景代码用 `allclose` 并改成 float64。
2. `kv_gib` 里 MQA 的 0.140625 GiB 就是 144 MiB，换单位时注意 1 GiB = 1024 MiB。
3. MLA 紧凑缓存的 576 维里，64 维是额外分支；只算 512 维会少算 11%。

## 4. 自己改一改

- 把 `kv_gib` 的上下文从 8192 改成 131072（128K），看三种结构各多大。
- 随机生成更大的 `c`、`U`、`q`，验证两条路径仍然相等。
- 比较紧凑路径每步的额外工作：给 1 个查询做 $128 \times 512$ 的变换，和给 8192 个 token 都升维相比差多少倍。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 逐页笔记随书厚，摘要一张常改写；
> 打印机少人不少，KV 份数减、查询头不减；
> 压缩包里存潜变量，结合律把升维挪到查询上。